In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from tqdm import tqdm
import re

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, PeftModel
import json
import torch
import matplotlib.pyplot as plt

In [ ]:
import random
random.seed(42)

In [ ]:
split = "llm-gold-train"

# annotation_type = "CRAC_provided_plaintext"
# annotation_type = "XML_tags"
# annotation_type = "simplified_XML_tags"
annotation_type = "head_only" # XML_tags


training_datasets = ["all"]
# training_datasets = ["all", "ca_ancora-corefud-train"]

reindex_coref_ids = True

# model_name = "google/gemma-3-270m-it"
# model_name = "google/gemma-3-1b-it"
# model_name = "google/gemma-3-4b-it"
# model_name = "google/gemma-3-12b-it"
model_name = "google/gemma-3-27b-it"

max_epochs = 4

if model_name == "google/gemma-3-270m-it":
    train_config = {"lora_rank": 16,
                    "lora_alpha": 32,
                    "lora_dropout": 0.05,
                    "per_device_train_batch_size": 2,
                    "gradient_accumulation_steps": 16,
                    "learning_rate": 2e-4,
                    "max_seq_length": 4096,
                    "sentence_batch": 4,
                    "previous_context_tokens_length": 250}
if model_name == "google/gemma-3-1b-it":
    train_config = {"lora_rank": 16,
                    "lora_alpha": 32,
                    "lora_dropout": 0.05,
                    "per_device_train_batch_size": 2,
                    "gradient_accumulation_steps": 16,
                    "learning_rate": 2e-4,
                    "max_seq_length": 4096,
                    "sentence_batch": 4,
                    "previous_context_tokens_length": 250}
if model_name == "google/gemma-3-4b-it":
    train_config = {"lora_rank": 16,
                    "lora_alpha": 32,
                    "lora_dropout": 0.05,
                    "per_device_train_batch_size": 4,
                    "gradient_accumulation_steps": 8,
                    "learning_rate": 1e-4,
                    "max_seq_length": 4096,
                    "sentence_batch": 4,
                    "previous_context_tokens_length": 250}
elif model_name == "google/gemma-3-12b-it":
    train_config = {
        "lora_rank": 64,  # Increased for a 27B parameter space
        "lora_alpha": 128,  # Alpha = 2 * Rank
        "lora_dropout": 0.05,
        "per_device_train_batch_size": 2,  # You have 96GB; use it to increase stability
        "gradient_accumulation_steps": 16,  # Total batch size stays 64 (4 * 2 GPUs * 8)
        "learning_rate": 5e-5,  # Slightly lower; big models are more sensitive
        "max_seq_length": 8192,  # 27B shines with more context
        "sentence_batch": 6,  # Can process more sentences per example
        "previous_context_tokens_length": 1024  # Catch long-range coreference chains
    }
elif model_name == "google/gemma-3-27b-it":
    train_config = {
        "lora_rank": 64,  # Increased for a 27B parameter space
        "lora_alpha": 128,  # Alpha = 2 * Rank
        "lora_dropout": 0.01,
        "per_device_train_batch_size": 1,  # 2 You have 96GB; use it to increase stability
        "gradient_accumulation_steps": 32,  # 16 Total batch size stays 64 (4 * 2 GPUs * 8)
        "learning_rate": 5e-5,  # Slightly lower; big models are more sensitive
        "max_seq_length": 8192,  # 27B shines with more context
        "sentence_batch": 6,  # Can process more sentences per example
        "previous_context_tokens_length": 1024  # Catch long-range coreference chains
    }

sentence_batch = train_config["sentence_batch"]
previous_context_tokens_length = train_config["previous_context_tokens_length"]


if model_name.startswith("google/gemma-3"):
    instruction_template = "<start_of_turn>user\n"
    response_template = "<start_of_turn>model\n"


if model_name.startswith("google/gemma-4"):
    instruction_template = "<|turn>user\n"
    response_template = "<|turn>model\n"



CRAC25_baseline_annotated_plaintext_path = f"/data-lachesis/common/shared_tasks/crac26/annotated_plaintext/{split}_{annotation_type}"
with open(CRAC25_baseline_annotated_plaintext_path, "r", encoding="utf-8") as f:
    CRAC25_baseline_annotated_plaintext = json.load(f)

print(CRAC25_baseline_annotated_plaintext_path)

In [ ]:
CRAC25_baseline_annotated_plaintext.keys()

In [ ]:
def reindex_previous_context(global_index_text, annotation_type="simplified_XML_tags"):
    if  annotation_type == "simplified_XML_tags":
        ent_opening_prefix="<ent"
        zero_prefix="</zero"
    elif annotation_type == "head_only":
        ent_opening_prefix="<ent"
        zero_prefix="<zero"

    text = global_index_text
    while isinstance(text, list):
        text = " ".join(text)

    # Escape prefixes for regex safety
    ent_escaped = re.escape(ent_opening_prefix)
    zero_escaped = re.escape(zero_prefix)

    # Build dynamic regex pattern
    pattern = rf"(?:{ent_escaped}|{zero_escaped})(\d+)>"

    # Extract IDs in order of appearance
    COREF_ids = re.findall(pattern, text)

    # Build ordered mapping
    mapping = {}
    for global_id in COREF_ids:
        if global_id not in mapping:
            mapping[global_id] = str(len(mapping))

    # Replacement function
    def replace_tag(match):
        full_prefix = match.group(0)[:match.group(0).find(match.group(1))]
        number = match.group(1)
        return f"{full_prefix}{mapping[number]}>"

    if isinstance(global_index_text, list):
        reindexed_text = []
        for text in global_index_text:
            reindexed_text.append(re.sub(pattern, replace_tag, text))

    else:
        reindexed_text = re.sub(pattern, replace_tag, text)

    return reindexed_text

In [ ]:
def generate_prompt_user_content_prompt(previous_context_text, input_text, annotation_type="simplified_XML_tags"):
# Fixed menu of possible IDs to ground the model

    if annotation_type == "simplified_XML_tags":
        tag_examples = ("- Entities: <entID> ... </ent>\n"
                         "- Zeros: ... </zeroID>\n")
    elif annotation_type == "head_only":
        tag_examples = ("- Entities: TOKEN <entID>\n"
                         "- Zeros: TOKEN <zeroID>\n")


    prompt = (
        "### TASK: COREFERENCE ANNOTATION\n"
        "Annotate mentions and zero anaphora. Do not change input text.\n\n"
        "### ALLOWED TAGS\n"
        f"{tag_examples}"
        f"\n### PREVIOUS CONTEXT\n{previous_context_text}\n\n"
        f"### INPUT TO ANNOTATE\n{input_text}\n\n"
        "### ANNOTATED OUTPUT\n"
    )
    return prompt

print(generate_prompt_user_content_prompt("This is the previous context", "This is the input", annotation_type=annotation_type))

In [ ]:
training_examples_dict = {}
for training_dataset in training_datasets:
    training_examples = []

    if training_dataset == "all":
        dataset_names = list(CRAC25_baseline_annotated_plaintext.keys())
    else:
        dataset_names = [training_dataset]

    for dataset_name, dataset_dict in tqdm(list(CRAC25_baseline_annotated_plaintext.items())[:]):
        if dataset_name not in dataset_names:
            continue
        for document in dataset_dict:
            raw_sentences = [sentence["input"] for sentence in document]
            annotated_sentences = [sentence["gold"] for sentence in document]

            min_sentence_id, max_sentence_id = 0, 0
            previous_context_tokens = []

            while max_sentence_id < len(document):
                max_sentence_id = min(min_sentence_id+sentence_batch, len(document))
                input_sentences = raw_sentences[min_sentence_id:max_sentence_id]
                input_tokens = [item for sublist in input_sentences for item in sublist]
                output_sentences = annotated_sentences[min_sentence_id:max_sentence_id]
                output_tokens = [item for sublist in output_sentences for item in sublist]

                min_sentence_id = max_sentence_id

                previous_context_text = " ".join(previous_context_tokens) if len(previous_context_tokens) > 0 else "[None]"
                input_text = " ".join(input_tokens)
                output_text = " ".join(output_tokens)

                if reindex_coref_ids:
                    previous_context_text, output_text = reindex_previous_context([previous_context_text, output_text], annotation_type=annotation_type)

                previous_context_tokens += output_tokens
                previous_context_tokens = previous_context_tokens[-previous_context_tokens_length:]

                prompt_user_content = generate_prompt_user_content_prompt(previous_context_text, input_text, annotation_type=annotation_type)
                training_example = {
                    # The 'prompt' is the user part of the conversation
                    "prompt": [
                        {"role": "user", "content": prompt_user_content}
                    ],
                    # The 'completion' is the assistant part
                    "completion": [
                        {"role": "assistant", "content": output_text}
                    ],
                }
                training_examples.append(training_example)

    random.shuffle(training_examples)
    training_examples_dict[training_dataset] = training_examples

for training_dataset, training_examples in training_examples_dict.items():
    print(f"{training_dataset!r}: {len(training_examples):,} training examples")

In [ ]:
all_training_examples = [
    example
    for training_examples in training_examples_dict.values()
    for example in training_examples
]

In [ ]:
training_example["completion"][0]["content"]

In [ ]:
all_used_coref_ids = []
for training_example in all_training_examples:
    pattern = r"<(?:ent|/zero)(\d+)>"
    found_ids = re.findall(pattern, training_example["completion"][0]["content"])
    # Convert to ints, remove duplicates, and sort
    unique_ids = sorted(list(set(int(i) for i in found_ids)))
    all_used_coref_ids.extend(unique_ids)
max(all_used_coref_ids)

In [ ]:
dataset = Dataset.from_list(all_training_examples[:])

dataset[0]

In [ ]:
print(f"Training Examples: {len(dataset):,}")

In [ ]:
print(model_name.split("/")[-1])

trained_model_directory = "#_trained_models"
os.makedirs(trained_model_directory, exist_ok=True)

trained_model_name = f"{model_name.split('/')[-1]}_TrainContext_{previous_context_tokens_length}_TrainSentBatch_{sentence_batch}_Tag_{annotation_type}_Reindex_{reindex_coref_ids}_ValidCOREF_False_{'|'.join(training_datasets)}_multi_epochs"

output_directory = os.path.join(trained_model_directory, trained_model_name)
print(f"Output directory : {output_directory}")

In [ ]:
# 2. Quantization 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 3. Load model and tokenizer

# After loading tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, return_token_ids=True)
if tokenizer.pad_token is None:
    print("Assigning padding token")
    tokenizer.pad_token = tokenizer.eos_token


# Force tokenizer to return token_type_ids
tokenizer.model_input_names = ["input_ids", "attention_mask", "token_type_ids"]

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    # attn_implementation="flash_attention_2"  # si supporté
    attn_implementation="eager"  # si supporté
)

# 4. Gradient checkpointing
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()

# 5. LoRA
peft_config = LoraConfig(
    r=train_config["lora_rank"],
    lora_alpha=train_config["lora_alpha"],
    lora_dropout=train_config["lora_dropout"],
    # target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    target_modules="all-linear",
    # target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    # modules_to_save=["lm_head", "embed_token"],
    task_type="CAUSAL_LM",
)

from transformers import TrainerCallback

class SaveEpochCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(state.epoch)
        kwargs["model"].save_pretrained(f"{args.output_dir}/epoch-{epoch}")
        if "tokenizer" in kwargs:
            kwargs["tokenizer"].save_pretrained(f"{args.output_dir}/epoch-{epoch}")

# 6. Configuration entraînement
# 1. Update SFTConfig (Remove max_seq_length from here)
training_args = SFTConfig(
    output_dir=output_directory,
    per_device_train_batch_size=train_config["per_device_train_batch_size"], # 4
    gradient_accumulation_steps=train_config["gradient_accumulation_steps"], # 8
    num_train_epochs=max_epochs,
    learning_rate=train_config["learning_rate"],
    # --- STRATEGY UPDATE ---
    save_strategy="steps",  # Save exactly once per full pass
    save_steps=25,
    save_total_limit=10,  # Keep all 5 epochs to compare later
    logging_strategy="steps",
    logging_steps=5,  # Vital for tiny datasets to see every move
    # New logic for masking
    # dataset_text_field="text",  # Tell it to look at your pre-formatted column
    # packing=False,              # Packing is incompatible with completion_only_loss
    completion_only_loss=True,
    dataset_kwargs={
        "add_special_tokens": True,
        # Templates MUST go here, inside dataset_kwargs
        # "instruction_template": "<start_of_turn>user\n",
        # "response_template": "<start_of_turn>model\n",
    },
    # Rest of your args
    bf16=True,
    lr_scheduler_type="linear",


    remove_unused_columns=False, # Important for Gemma 3 token_type_ids
    max_length=train_config["max_seq_length"], # 3072, 4096
    optim="paged_adamw_8bit",  # 8-bit optimizer saves ~30% optimizer state memory
)

class Gemma3SFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        if "token_type_ids" not in inputs:
            inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])

        # --- MASKING CHECK ---
        if not hasattr(self, "_mask_checked"):
            labels = inputs["labels"]
            # Count tokens where loss is calculated (labels != -100)
            active_loss_tokens = (labels != -100).sum().item()
            total_tokens = labels.numel()
            print(f"\n[DEBUG] Active loss tokens: {active_loss_tokens} / {total_tokens}")

            # If active_loss_tokens is almost equal to total_tokens,
            # masking is NOT working.
            if active_loss_tokens > (total_tokens * 0.8):
                print("⚠️ WARNING: Loss is being calculated on the Prompt! Check your templates.")
            else:
                print("✅ SUCCESS: Prompt is correctly masked.")
            self._mask_checked = True
        # ---------------------

        return super().compute_loss(model, inputs, return_outputs, num_items_in_batch)


# 2. Pass max_seq_length to the Trainer directly
trainer = Gemma3SFTTrainer(
    model=model,
    train_dataset=dataset, # Now has 'prompt' and 'completion' columns
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.add_callback(SaveEpochCallback())

In [ ]:
checkpoint_dir = training_args.output_dir
last_checkpoint = None
if os.path.isdir(checkpoint_dir):
    checkpoints = [os.path.join(checkpoint_dir, d) for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint")]
    if checkpoints:
        last_checkpoint = sorted(checkpoints)[-1]  # dernier checkpoint trouvé

if last_checkpoint:
    print(f"Starting from checkpoint: {last_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint. Starting from scratch.")
    train_result = trainer.train()

trainer.save_model()

In [ ]:
# 8. Ploting Loss
log_history = trainer.state.log_history
losses = [log["loss"] for log in log_history if "loss" in log]
steps = [log["step"] for log in log_history if "loss" in log]

os.makedirs("logs", exist_ok=True)

plt.figure(figsize=(10, 6))
plt.plot(steps, losses, label="Coreference Resolution Finetuning Training Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.grid(True)
plt.savefig(f"logs/{trained_model_name}.png")
plt.show()